In [109]:
# Import the required libraries
import pandas as pd
import numpy as np

In [110]:
# Load the data 
data = pd.read_csv('Diabetes.csv')
data.head()

,Glucose,BloodPressure,Diabetes
0,148,72,1
1,85,66,0
2,183,64,1
3,89,66,0
4,137,40,1


In [111]:
# Calculate entropy function
def calculate_entropy(data_label):
    classes,class_counts = np.unique(data_label,return_counts = True)
    entropy_value = np.sum([(-class_counts[i]/np.sum(class_counts))*np.log2(class_counts[i]/np.sum(class_counts)) 
                        for i in range(len(classes))])
    return entropy_value
# Calculate information gain function
def calculate_information_gain(dataset,feature,label): 
    # Calculate the dataset entropy
    dataset_entropy = calculate_entropy(dataset[label])   
    values,feat_counts= np.unique(dataset[feature],return_counts=True)
    
    # Calculate the weighted feature entropy                                # Call the calculate_entropy function
    weighted_feature_entropy = np.sum([(feat_counts[i]/np.sum(feat_counts))*calculate_entropy(dataset.where(dataset[feature]
                              ==values[i]).dropna()[label]) for i in range(len(values))])    
    feature_info_gain = dataset_entropy - weighted_feature_entropy
    return feature_info_gain

In [112]:
# Set the features and label
features = data.columns[:-1]
label = 'Diabetes'
parent=None
features

Index(['Glucose', 'BloodPressure'], dtype='object')

In [113]:
# Defining the decision_tree_function
def decision_tree_function(dataset, data, features,label, parent):
    
    datum = np.unique(data[label], return_counts = True)
    unique_data = np.unique(dataset[label])
    
    if len(unique_data) <= 1:
        return unique_data[0]
    
    elif len(dataset) == 0:
        return unique_data[np.argmax(datum[1])]
    
    elif len(features) == 0:
        return parent
    
    else:
        parent = unique_data[np.argmax(datum[1])]
        
        # Call the calculate_information_gain function
        item_values = [calculate_information_gain(dataset, feature, label) for feature in features]
        
        optimum_feature_index = np.argmax(item_values)
        optimum_feature = features[optimum_feature_index]
        decision_tree = {optimum_feature:{}}
        features = [i for i in features if i!= optimum_feature] 
        
        for value in np.unique(dataset[optimum_feature]):
            min_data = dataset.where(dataset[optimum_feature] == value).dropna()
            
            # Recursive call to decision_tree_function function
            min_tree = decision_tree_function(min_data, data, features, label, parent)
            
            decision_tree[optimum_feature][value] = min_tree
            
        return(decision_tree)

In [114]:
# Define the predict function
def predict_diabetes(test_data, decision_tree):
    
    for nodes in decision_tree.keys():
        value = test_data[nodes]
        decision_tree = decision_tree[nodes][value]
        
        prediction = 0
        if type(decision_tree) is dict:
            
            # Recursive call to predict_diabetes function
            prediction = predict_diabetes(test_data, decision_tree)
        
        else:
            prediction = decision_tree
            break;
            
    return prediction

In [115]:
# Set the features and label
features = data.columns[:-1]
label = 'Diabetes'
parent=None
features

Index(['Glucose', 'BloodPressure'], dtype='object')

In [116]:
# Train the decision tree model
decision_tree = decision_tree_function(data,data,features,label,parent)

In [120]:
# Predict using the trained model
sample_data = {'Glucose':148,'BloodPressure':72}
test_data = pd.Series(sample_data)
prediction = predict_diabetes(test_data,decision_tree)
prediction

1.0